# Projeto Final — RideSmart
## Modelagem e Análise de Rotas Urbanas com Grafos


### Resumo do problema

Dado um ponto de origem `A`, um destino `B` e uma distância máxima de caminhada `X`, o objetivo é
escolher um **ponto de embarque** `P` (a no máximo `X` metros de caminhada de `A`) que minimize o
custo total da viagem, composta por dois trechos:

```text
A → P   (a pé)
P → B   (de carro)
```

O notebook modela a malha viária real como grafo, implementa e compara quatro algoritmos de caminho
mínimo, introduz **trânsito sintético** e analisa o *trade-off* entre caminhar um pouco e chegar
mais rápido.

## 1. Modelagem do problema como grafo

**Grafo dirigido com multiarestas** $G = (V, E)$ obtido do OpenStreetMap via OSMnx:

- **Nós ($V$):** cruzamentos e extremidades de vias. Cada nó possui coordenadas geográficas
  (`y` = latitude, `x` = longitude).
- **Arestas ($E$):** segmentos de rua entre dois cruzamentos. O grafo é um `MultiDiGraph`:
  é **dirigido** (modela ruas de mão única) e admite **arestas paralelas** (duas ruas distintas
  ligando o mesmo par de cruzamentos).

**Funções de custo (pesos das arestas):**

| Peso | Significado | Usado para |
|---|---|---|
| `length` | comprimento do segmento em metros | menor **distância** e caminhada `A→P` |
| `travel_time` | tempo de percurso de carro em segundos (sem trânsito) | rota mais **rápida** |
| `travel_time_traffic` | `travel_time` ajustado por um fator de congestionamento | rota mais rápida **com trânsito** |

O tempo de caminhada é derivado do comprimento por uma velocidade fixa de pedestre
($v_{pe} \approx 1{,}4$ m/s $\approx 5$ km/h):
$$t_{pe}(A \to P) = \frac{\text{distância}_{length}(A \to P)}{v_{pe}}.$$

O **custo total** de uma escolha de embarque `P` é:
$$\text{custo}(P) = t_{pe}(A \to P) + \text{custo}_{carro}(P \to B),$$
sujeito à restrição $\text{distância}_{length}(A \to P) \le X$.

## 2. Configuração do ambiente

Em ambientes como o Google Colab, descomente a instalação abaixo. O OSMnx requer acesso à internet
para baixar os dados do OpenStreetMap.

In [1]:
!pip install osmnx networkx pandas matplotlib scikit-learn

import time, copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

ox.settings.use_cache = True          # cacheia o download (re-execuções ficam instantâneas)
ox.settings.log_console = False
print("OSMnx", ox.__version__, "| NetworkX", nx.__version__)

OSMnx 2.1.0 | NetworkX 3.6.1


## 3. Download da malha viária real

Baixamos a rede **dirigível** (`network_type="drive"`) num raio em torno de um ponto central da
cidade. Em seguida o OSMnx imputa velocidades por tipo de via (`add_edge_speeds`) e calcula o tempo
de percurso de cada aresta (`add_edge_travel_times`).

In [2]:
# Centro da área de estudo (exemplo: bairro de Petrópolis / Tirol, Natal-RN).
# Troque pelas coordenadas da sua cidade/bairro.
CENTRO = (-5.8000, -35.2050)   # (lat, lon)
RAIO_M = 2500                  # raio da malha em metros

G = ox.graph_from_point(CENTRO, dist=RAIO_M, network_type="drive")

# velocidades (km/h) e tempos de percurso (s) por aresta
try:
    G = ox.routing.add_edge_speeds(G)
    G = ox.routing.add_edge_travel_times(G)
except AttributeError:                 # compatibilidade com versões antigas
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

print(f"[drive] Nós: {G.number_of_nodes()}  |  Arestas: {G.number_of_edges()}")
u, v, k, d = list(G.edges(keys=True, data=True))[0]
print(f"  Aresta exemplo: {d.get('name')} | length={d.get('length'):.1f}m | travel_time={d.get('travel_time'):.1f}s")

# Grafo separado para caminhada (inclui calçadas e vias de pedestre)
G_walk = ox.graph_from_point(CENTRO, dist=RAIO_M, network_type="walk")
print(f"[walk]  Nós: {G_walk.number_of_nodes()}  |  Arestas: {G_walk.number_of_edges()}")

[drive] Nós: 3104  |  Arestas: 6762
  Aresta exemplo: Avenida Governador Sílvio Pedroza | length=28.2m | travel_time=1.9s


[walk]  Nós: 4932  |  Arestas: 14462


## 4. Definição de origem `A`, destino `B`, distância máxima `X`

Escolhemos coordenadas reais de `A` e `B` e mapeamos cada uma para o nó mais próximo da malha
(`nearest_nodes`). `X` é a distância máxima que o usuário aceita caminhar; `V_PE` é a velocidade de
caminhada.

In [3]:
# Coordenadas de origem e destino (lat, lon) — ajuste conforme sua cidade
A_latlon = (-5.7935, -35.2010)
B_latlon = (-5.8120, -35.1980)

X      = 600.0    # distância máxima de caminhada (metros)
V_PE   = 1.4      # velocidade de caminhada (m/s) ~ 5 km/h

A = ox.distance.nearest_nodes(G, A_latlon[1], A_latlon[0])
B = ox.distance.nearest_nodes(G, B_latlon[1], B_latlon[0])
A_walk = ox.distance.nearest_nodes(G_walk, A_latlon[1], A_latlon[0])
print(f"Nó de origem  A = {A}  (walk: {A_walk})")
print(f"Nó de destino B = {B}")

# velocidade máxima do grafo (m/s) — usada na heurística do A* para tempo
VMAX_MS = max(d.get('speed_kph', 1) for _,_,d in G.edges(data=True)) / 3.6
print(f"Velocidade máxima do grafo: {VMAX_MS*3.6:.0f} km/h")

Nó de origem  A = 8626298111  (walk: 7616423477)
Nó de destino B = 503422480
Velocidade máxima do grafo: 70 km/h


## 5. Implementação dos algoritmos

Nesta seção implementaremos quatro algoritmos de caminho mínimo:

1. **Dijkstra simples** — implementação clássica usando NetworkX (`nx.dijkstra_path`)
2. **Dijkstra com fila de prioridade (heap)** — implementação manual com `heapq` do Python
3. **A* com heurística geográfica** — A* usando distância euclidiana como heurística
4. **Bellman-Ford** — algoritmo robusto que lida com arestas negativas

Cada algoritmo será:
- Implementado e testado com pelo menos 2 pares de nós
- Medido em tempo de execução e nós expandidos
- Coletado para comparação posterior

## 6. Trânsito sintético

Modelamos o trânsito aplicando fatores de congestionamento baseados no tipo de via:

| Categoria de via | Fator de congestionamento |
|---|---|
| Arterial (motorway, trunk, primary) | 2.0 |
| Coletora (secondary, tertiary) | 1.5 |
| Residencial (residential, living_street) | 1.1 |

O trânsito sintético cria uma nova aresta `travel_time_traffic` = `travel_time` × fator,
permitindo comparar rotas com e sem congestionamento.

## 7. Candidatos de embarque

Para encontrar todos os pontos de embarque `P` viáveis:

1. Executamos Dijkstra no grafo de caminhada (`G_walk`) a partir de `A`
2. Calculamos a distância real ao longo das ruas (não euclidiana)
3. Filtramos candidatos com distância ≤ `X` metros

Isso garante que cada candidato `P` é alcançável a pé dentro da restrição do usuário.

## 8. Função de avaliação

A função `avaliar_rota(G, G_walk, A, B, P, dist_walk, traffic_map)` encapsula:

- Execução dos 4 algoritmos no grafo de carro (`G`)
- Cálculo de distância e tempo para cada algoritmo
- Inclusão do tempo de caminhada `A → P`
- Retorno de dict com todas as métricas para um candidato `P`

Essa função será chamada para cada candidato, gerando o DataFrame de comparação.

## 9. Comparação e DataFrame

Com as métricas coletadas, geramos um DataFrame pandas com:

- Uma linha por candidato `P`
- Colunas: distância walk, distância carro, tempo total, tempo com trânsito, algoritmo usado

As **5 comparações obrigatórias**:
1. Menor rota em distância
2. Rota mais rápida sem trânsito
3. Rota mais rápida com trânsito sintético
4. Caso sem caminhada (A → B direto)
5. Ganho obtido ao caminhar até outro ponto de embarque

## 10. Visualizações

Quatro tipos de visualização:

1. **Mapa estático** — rede viária + candidatos P + rotas escolhidas
2. **Gráfico de barras** — comparar tempo/distância entre os 4 algoritmos
3. **Gráfico de dispersão** — trade-off caminhar vs tempo total
4. **Tabela estilizada** — DataFrame com gradientes de cor

## 11. Animações

Duas abordagens de animação:

### Matplotlib GIF
- 4 GIFs individuais (um por algoritmo) mostrando expansão de nós
- 1 GIF comparativo (lado a lado ou sequencial)

### Plotly interativo
- Mapa interativo com candidatos P + rotas
- Zoom/pan no navegador
- Exporta HTML standalone

## 12. Análise dos resultados

Discussão das 12 questões do projeto:

1. Como o problema foi modelado como grafo?
2. O que representam nós e arestas?
3. Quais pesos foram usados?
4. Como trânsito sintético alterou rotas?
5. Caminhar melhorou solução?
6. Em quais casos caminhar atrapalhou?
7. Menor distância = rota mais rápida?
8. A* expandiu menos nós que Dijkstra?
9. Dijkstra heap mais eficiente que simples?
10. Bellman-Ford trouxe ganho?
11. Limitações da modelagem?
12. Como aproximar de app real?

## 13. Conclusão

Síntese dos principais achados:

- Qual algoritmo foi mais eficiente?
- O trânsito sintético mudou a escolha do candidato P?
- Vale a pena caminhar para um ponto melhor?
- Limitações e trabalhos futuros